### Модуль подготовки датасета для задачи Keyword Spotting.
#### Извлечение призНаков аудиосигнала, разбиение на выборки и сериализация результата.


In [3]:
import librosa
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from pathlib import Path
from tqdm import tqdm
import numpy as np


def extract_mfcc_with_deltas(file_path, sr=16000, n_mfcc=13, max_len=101):
    """
    Извлечение MFCC признаков и их производных из аудиофайла.
    Возвращает массив формы (max_len, n_mfcc*3).
    
    Аргументы:
        file_path (str): Путь к .wav файлу.
        sr (int): Частота дискретизации.
        n_mfcc (int): Количество MFCC коэффициентов.
        max_len (int): Желаемая длина временной оси (фреймов).
    
    Возвращает:
        np.ndarray | None: Массива признаков или None при ошибке обработки.
    """
    try:
        # Загрузка аудиофайла с преобразованием частоты дискретизации
        audio, _ = librosa.load(file_path, sr=sr)
        
        # Нормализация громкости для обеспечения устойчивости признаков
        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val

        # Извлечение MFCC-коэффициентов
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        
        # Вычисление первой и второй производных (дельта-признаки)
        delta  = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        
        # Объединение базовых MFCC и их производных
        features = np.vstack([mfcc, delta, delta2])

        # Приведение к фиксированной длине: обрезка либо дополнение нулями
        if features.shape[1] > max_len:
            features = features[:, :max_len]
        else:
            pad_width = max_len - features.shape[1]
            features = np.pad(features, ((0, 0), (0, pad_width)), mode='constant')

        # Транспонирование: (n_features, time) => (time, n_features)
        return features.T

    except Exception as e:
        # Журналируем ошибку, возвращаем None - файл будет пропущен из датасета
        print(f"[ERROR] Ошибка при обработке файла {file_path}: {e}")
        return None

def load_dataset(positive_dir, negative_dir, max_len=101, sr=16000, n_mfcc=13):
    """
    Загрузка и подготовка датасета из папок с положительными и отрицательными примерами.
    
    Аргументы:
        positive_dir (str): Путь к папке с положительными примерами (.wav).
        negative_dir (str): Путь к папке с отрицательными примерами (.wav).
        max_len (int): Максимальная длина временной последовательности (фреймов).
        sr (int): Частота дискретизации.
        n_mfcc (int): Количество MFCC коэффициентов в каждом фрейме.
    
    Возвращает:
        X (np.ndarray): Массив признаков формы (n_samples, max_len, n_mfcc*3).
        y_categorical (np.ndarray): Метки классов (one-hot) формы (n_samples, 2).
        y (np.ndarray): Целочисленные метки классов (0/1).
    """
    # Получение списка файлов соответствующих классов
    positive_files = list(Path(positive_dir).glob("*.wav"))
    negative_files = list(Path(negative_dir).glob("*.wav"))

    print(f"[INFO] Найдено положительных примеров: {len(positive_files)}")
    print(f"[INFO] Найдено отрицательных примеров: {len(negative_files)}")

    X = []
    y = []

    # Загрузка и обработка положительных примеров
    print("[INFO] Загрузка положительных примеров...")
    for file_path in tqdm(positive_files, desc="Positive"):
        features = extract_mfcc_with_deltas(str(file_path), sr=sr, n_mfcc=n_mfcc, max_len=max_len)
        if features is not None:
            X.append(features)
            y.append(1)  # Класс 1 - целевая фраза

    # Загрузка и обработка отрицательных примеров
    print("[INFO] Загрузка отрицательных примеров...")
    for file_path in tqdm(negative_files, desc="Negative"):
        features = extract_mfcc_with_deltas(str(file_path), sr=sr, n_mfcc=n_mfcc, max_len=max_len)
        if features is not None:
            X.append(features)
            y.append(0)  # Класс 0 - нецелевая фраза

    # Преобразование данных к формату numpy с жестким типом для экономии памяти
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)

    # Перемешивание индексов для исключения порядка классов в датасете
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    X = X[indices]
    y = y[indices]

    # Преобразование меток в one-hot-кодировку для задачи классификации
    y_categorical = to_categorical(y, num_classes=2)

    print(f"\nИтоговая размерность X: {X.shape}")
    print(f"Итоговая размерность y (one-hot): {y_categorical.shape}")
    print(f"Соотношение классов: 0 = {np.sum(y == 0)}, 1 = {np.sum(y == 1)}")
    print(f"Соотношение (neg/pos): {np.sum(y == 0)/max(np.sum(y == 1),1):.2f}:1")
    return X, y_categorical, y

In [ ]:
POSITIVE_DIR = "D:/Аудио выборка/positive_frames"   # Путь к директории положительных примеров
NEGATIVE_DIR = "D:/Аудио выборка/negative_frames"   # Путь к директории отрицательных примеров

#### Загрузка и обработка полного датасета

In [4]:
X, y_cat, y_int = load_dataset(
    positive_dir=POSITIVE_DIR,
    negative_dir=NEGATIVE_DIR,
    max_len=101,
    sr=16000,
    n_mfcc=13
)

# Разделение на обучающую, валидационную и тестовую выборки

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.3, random_state=42, stratify=y_int
)  # Первое разбиение: тренировочная и временная (валидация+тест), с сохранением соотношения классов

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, 
    stratify=np.argmax(y_temp, axis=1)
)  # Второе разбиение: из временной выборки выделяем тестовую и валидационную части


print(f"\nРазмер обучающей выборки: {X_train.shape}")
print(f"Размер валидационной выборки: {X_val.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Найдено положительных примеров: 14147
Найдено отрицательных примеров: 24183
Загрузка положительных примеров...


Positive: 100%|██████████| 14147/14147 [01:35<00:00, 148.59it/s]


Загрузка отрицательных примеров...


Negative: 100%|██████████| 24183/24183 [02:12<00:00, 181.91it/s]



Итоговая размерность X: (38330, 101, 39)
Итоговая размерность y: (38330, 2)
Соотношение классов: 0: 24183, 1: 14147
Соотношение (отр/пол): 1.71:1

Размер обучающей выборки: (26831, 101, 39)
Размер валидационной выборки: (5749, 101, 39)
Размер тестовой выборки: (5750, 101, 39)


####  Сохранение предобработанных датасетов для последующей работы

In [5]:
np.savez_compressed(
    "D:/Аудио выборка/kws_dataset.npz",  # путь сохранения
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test
)

print("\nДатасет сохранён в файл kws_dataset.npz")


Датасет сохранён в файл kws_dataset.npz
